# Checkpoint 2: RQ Formation


## Goal: Define research questions that require both course techniques and externally learned techniques.




### Project Scope
Brief recap of dataset and EDA findings – Ex:
- Dataset: Retail transactions
- Course techniques: Frequent itemsets
- External techniques: Sequential pattern mining
----


### Dataset -  Food.com



#### Dataset size and structure

* It contains 231,637 recipes with ingredient lists, recipe names, tags, nutrition metadata, descriptions, and cooking steps. Each recipe can be viewed both as an ingredient basket and as a text document. It also contains 718,379 user recipe interactions. The interaction side is user item rating data, and the two are linked by recipe identifiers.

#### Course Techniques

* Frequent Itemset Mining / Association Rules: treat ingredients as a basket and mine co-occurring ingredient sets.
* Graph mining - Ingredients can be modeled as a co-occurrence graph where nodes are ingredients and edges connect ingredients that appear in the same recipe. This supports centrality, PageRank, Personalized PageRank, and community detection

#### External Techniques

* Topic modeling on recipe descriptions and steps.
* Bipartite graph embeddings.


#### EDA recap
In project checkpoint 1, we explored both the interaction side and the recipe side of the Food.com dataset to understand what kind of structure the data actually provides. We first looked at how the dataset is organized in both its raw and preprocessed forms. The raw tables connect recipes and interactions directly through recipe id, while the preprocessed tables use dense user and recipe indices u and i to link the interaction splits with PP_users and PP_recipes. From this, we confirmed that the schema is consistent and that the dataset supports analysis from both the interaction perspective and the recipe-content perspective.

On the interaction side, combining the train, validation, and test splits gave 718,379 total interactions across 25,076 users and 178,265 recipes. One of the main findings from checkpoint 1 was that this user–recipe matrix is extremely sparse, with density about 0.0001607. Most users interact with only a small number of recipes, and most recipes receive only a small number of interactions. The median user has 7 interactions, while the median recipe has only 2. We also observed strong long-tail behavior in both user activity and recipe popularity. Ratings are heavily skewed toward high values, especially 4 and 5, and there are also 18,000 cases with rating 0, which likely need to be treated carefully rather than interpreted as a normal rating. These results showed that the interaction side is useful for understanding the recommendation structure of the dataset, but also that it is limited by sparse coverage.
When we moved to the recipe side, the dataset looked much more promising for mining structure directly from recipe content. Across 231,637 recipes, ingredient information was complete, with no missing or invalid ingredient rows in the processed ingredient view we examined. We also found that recipes are moderate in size: the average basket size is about 9 ingredients, the median is also 9, and 90 percent of recipes have 14 or fewer ingredients. This is useful because recipes are large enough to contain meaningful combinations, but still small enough that recipe-level processing remains manageable.

Another important observation from checkpoint 1 was that the ingredient space is large but highly imbalanced. There are 14,942 distinct ingredient strings and about 2.1 million total ingredient occurrences across recipes. However, a relatively small set of staple ingredients dominates the dataset. The top 100 ingredients account for about 51.9 percent of all ingredient occurrences, and the top 1,000 account for about 86.0 percent. Ingredients such as salt, butter, sugar, onion, and water appear across a large fraction of recipes, while many other ingredients are much rarer. So the recipe side also has a strong head-tail structure, but in a way that is directly meaningful for ingredient-based mining.

Based on these observations, checkpoint 1 gave us two useful conclusions. First, the interaction side helped show why a purely user-recipe based direction may be limited, mainly because of extreme sparsity and long-tail coverage. Second, the recipe side appeared cleaner, richer, and more suitable for mining meaningful patterns. At the same time, the ingredient frequency imbalance suggests that simple frequent itemset mining on the full vocabulary may mostly recover obvious staple combinations unless support thresholds are chosen very carefully. So moving forward, it seems more natural to continue in a recipe-focused direction while also considering methods that can better handle the long tail, such as ingredient co-occurrence analysis, graph-based methods, embeddings, or weighted representations that reduce the dominance of very common ingredients.

#### Licensing
* Kaggle dataset license is listed as Data files © Original Authors with author Shuyang Li- UCSD PhD researcher

In [8]:
import ast
import re
import math
import numpy as np
import pandas as pd

from collections import Counter
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import kneighbors_graph
from sklearn.decomposition import LatentDirichletAllocation

import networkx as nx
# For reproducibility
random.seed(42)
np.random.seed(42)

In [6]:
src = Path("kaggle.json")
assert src.exists(), "kaggle.json not found in the current directory (did upload work?)"

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

dst = kaggle_dir / "kaggle.json"
dst.write_bytes(src.read_bytes())
os.chmod(dst, 0o600)

print("Installed Kaggle credentials at:", dst)


DATASET = "shuyangli94/food-com-recipes-and-user-interactions"
OUT_DIR = Path("data/foodcom")
OUT_DIR.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d {DATASET} -p {OUT_DIR} --unzip
print("Downloaded into:", OUT_DIR.resolve())



Installed Kaggle credentials at: /root/.kaggle/kaggle.json
Dataset URL: https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions
License(s): copyright-authors
100% 267M/267M [00:01<00:00, 152MB/s]

Downloaded into: /content/data/foodcom


In [7]:
root = Path("data/foodcom")

train = pd.read_csv(root / "interactions_train.csv")
val   = pd.read_csv(root / "interactions_validation.csv")
test  = pd.read_csv(root / "interactions_test.csv")

pp_recipes = pd.read_csv(root / "PP_recipes.csv", low_memory=False)
pp_users   = pd.read_csv(root / "PP_users.csv", low_memory=False)

raw_recipes = pd.read_csv(root / "RAW_recipes.csv", low_memory=False)
raw_interactions = pd.read_csv(root / "RAW_interactions.csv", low_memory=False)

----

### Research Question Definition


## RQ1

**How much agreement is there between recipe clusters built from ingredient features and recipe clusters built from step-text features?**

This question is interesting because recipes may group differently depending on which part of the recipe we use. Two recipes can be similar in ingredients but different in preparation, or they can use different ingredients while following a similar cooking process. This question helps us test whether ingredient composition and procedural text recover the same recipe families or reveal different kinds of structure. It is also a good first question because it operates directly at the recipe level and uses two natural views of the same objects.

## RQ2

**Among recipes sharing a frequent ingredient itemset, what frequent step subsequences recur, and how concentrated are those procedural patterns?**

This question is interesting because shared ingredients do not automatically imply shared preparation. Some frequent ingredient combinations may correspond to one dominant procedural core, while others may appear across several different cooking procedures. This question connects the unordered basket view of ingredients with the ordered sequence view of recipe steps. It lets us ask not only which ingredients co-occur, but whether those common ingredient combinations are associated with recurring ordered procedure patterns.

## RQ3

**Do recipe groups induced by an ingredient-similarity graph align with latent text topics from recipe descriptions and steps?**

This question is interesting because ingredient-based structural similarity and text-based semantic similarity are not necessarily the same. A graph built from ingredient overlap may capture compositional similarity, while topics from recipe descriptions and steps may reflect dish type, cuisine style, or cooking intent. This question lets us compare whether graph structure and text semantics organize recipes in similar ways.

---

## RQ-to-method mapping table

| RQ  | Data mining task type                          | Course algorithms                                                                                                   | External algorithms                                                        |
| --- | ---------------------------------------------- | ------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------- |
| RQ1 | Clustering                                     |  Clustering on ingredient and step text vectors. K means and DBSCAN. | None                                                                       |
| RQ2 | Frequent itemset and sequential pattern mining | Apriori for associative rule mining                                                                                 | PrefixSpan or SPADE for sequential pattern mining on step action sequences |
| RQ3 | Graph mining and topic modeling                | Ingredient similarity graph and community detection using modularity based clustering                               | Topic modeling on descriptions and steps                                   |



In [11]:
def parse_list_col(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

def clean_text(s):
    s = s.lower()
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

ACTION_PATTERNS = [
    (r"\b(chop|dice|mince|slice|cut)\b", "chop"),
    (r"\b(mix|combine|stir|whisk|beat|blend|fold)\b", "mix"),
    (r"\b(saute|fry|stir fry)\b", "saute"),
    (r"\b(boil|simmer|poach)\b", "boil"),
    (r"\b(bake|roast)\b", "bake"),
    (r"\b(chill|cool|refrigerate|freeze)\b", "chill"),
    (r"\b(serve|garnish|plate)\b", "serve"),
]

def normalize_actions(step_list):
    actions = []
    for step in step_list:
        s = clean_text(step)
        found = []
        for pat, label in ACTION_PATTERNS:
            if re.search(pat, s):
                found.append(label)
        actions.extend(found)
    out = []
    for a in actions:
        if not out or out[-1] != a:
            out.append(a)
    return out

recipes = raw_recipes[["id", "name", "ingredients", "steps", "description"]].copy()
recipes["ingredients_list"] = recipes["ingredients"].apply(parse_list_col)
recipes["steps_list"] = recipes["steps"].apply(parse_list_col)
recipes = recipes[
    (recipes["ingredients_list"].str.len() > 0) &
    (recipes["steps_list"].str.len() > 0)
].copy()

recipes["step_text_clean"] = recipes["steps_list"].apply(lambda xs: clean_text(" ".join(xs)))
recipes["actions"] = recipes["steps_list"].apply(normalize_actions)
recipes["doc_text"] = (
    recipes["description"].fillna("").astype(str).str.lower() + " " +
    recipes["step_text_clean"]
).str.strip()

print(recipes.shape)

(231636, 10)


---

## Method and metric plan

### RQ1 plan


**Method**

Each recipe will be represented in two separate views. The first view is ingredient-based, using a recipe-by-ingredient representation such as a binary ingredient vector or a weighted ingredient vector that downweights very common staple ingredients. The second view is step-text-based, using a representation derived from the recipe instructions, such as TF-IDF over normalized step text or over extracted cooking-action tokens.

We will then cluster the same set of recipes in both views. The primary clustering algorithm will be hierarchical agglomerative clustering (HAC) with cosine distance, because both ingredient vectors and step-text vectors are sparse and high-dimensional, and cosine distance is more appropriate than raw Euclidean distance in that setting. HAC is also useful because it does not force spherical clusters and gives a hierarchy that may reveal coarse and fine recipe families. In addition, we will run k-means as a baseline course method, since it is simple, widely used, and provides a strong comparison point. DBSCAN can be used as an optional robustness check, but not as the main method, because density-based clustering is often harder to tune in sparse high-dimensional text-like spaces and may mark many recipes as noise.

After clustering both views, each recipe receives two cluster labels: one from the ingredient representation and one from the step-text representation. We then compare those two partitions of the same recipe set.

The comparison is between two different structural views of the same recipes. One partition reflects ingredient composition, while the other reflects procedural or instructional similarity. The goal is to test whether these two views produce similar recipe groupings or meaningfully different ones.

**Evaluation criteria**

The main agreement metrics will be Adjusted Rand Index (ARI) and Normalized Mutual Information (NMI), since both are standard ways to compare two clusterings of the same objects. We will also report an internal clustering quality measure such as silhouette score separately for each representation. Finally, we will inspect the top ingredients, top action words, or representative recipes in each cluster to evaluate interpretability.

**Motivation**

Clustering is explicitly part of the course, including k-means, hierarchical agglomerative clustering, DBSCAN, and clustering evaluation. Text Mining and vector-space style text representations are also covered, which makes this question strongly course-aligned.  
HAC is chosen as the primary algorithm because it is more natural than k-means for sparse, non-spherical recipe representations and gives more flexibility in examining cluster granularity. K-means is retained because it is simple, efficient. DBSCAN is not the main choice because recipe text and ingredient vectors are unlikely to form clean density-separated clusters in a stable way.

**Feasability**

This question is feasible because both clusterings are built on the same recipe objects using fields already available in the dataset. Ingredients can be represented directly from recipe ingredient lists, and step-side features can be derived from the recipe instructions after preprocessing. The methods are also practical to implement, since hierarchical clustering and k-means are standard and well-understood, and the comparison metrics are straightforward once both clusterings are produced.

**Risk**

The main risk is that clustering quality may depend strongly on the chosen feature representation and the number of clusters. Another risk is that step-text features may capture writing style or generic wording rather than true procedural similarity. There is also a scalability risk if hierarchical clustering is applied to a very large set of recipes without filtering or dimensionality reduction.


In [18]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import numpy as np
import pandas as pd

def identity(x):
    return x

rq1 = recipes.sample(2000, random_state=42).reset_index(drop=True)

# ingredient view
vec_ing = TfidfVectorizer(
    tokenizer=identity,
    preprocessor=identity,
    token_pattern=None,
    lowercase=False,
    min_df=10
)
X_ing = vec_ing.fit_transform(rq1["ingredients_list"])

# step-text view
vec_step = TfidfVectorizer(
    stop_words="english",
    min_df=5,
    max_df=0.6,
    ngram_range=(1, 2)
)
X_step = vec_step.fit_transform(rq1["step_text_clean"])

# remove zero rows in either view
mask = (X_ing.getnnz(axis=1) > 0) & (X_step.getnnz(axis=1) > 0)
rq1 = rq1.loc[mask].reset_index(drop=True)
X_ing = X_ing[mask]
X_step = X_step[mask]

# reduce dimensions
X_ing_red = TruncatedSVD(n_components=min(50, X_ing.shape[1] - 1), random_state=42).fit_transform(X_ing)
X_step_red = TruncatedSVD(n_components=min(50, X_step.shape[1] - 1), random_state=42).fit_transform(X_step)

k = 12
lab_ing = AgglomerativeClustering(n_clusters=k, metric="cosine", linkage="average").fit_predict(X_ing_red)
lab_step = AgglomerativeClustering(n_clusters=k, metric="cosine", linkage="average").fit_predict(X_step_red)

print("Rows kept:", len(rq1))
print("Ingredient matrix:", X_ing.shape)
print("Step-text matrix:", X_step.shape)
print("ARI:", adjusted_rand_score(lab_ing, lab_step))
print("NMI:", normalized_mutual_info_score(lab_ing, lab_step))

print("\nIngredient cluster sizes:")
print(pd.Series(lab_ing).value_counts().sort_index())

print("\nStep-text cluster sizes:")
print(pd.Series(lab_step).value_counts().sort_index())

Rows kept: 1974
Ingredient matrix: (1974, 315)
Step-text matrix: (1974, 4635)
ARI: 0.11800889947522501
NMI: 0.1357572203673072

Ingredient cluster sizes:
0     606
1     116
2     126
3      50
4     121
5     606
6      59
7      93
8      47
9      43
10     39
11     68
Name: count, dtype: int64

Step-text cluster sizes:
0     870
1      90
2      29
3     222
4      73
5     390
6      34
7      30
8      58
9      79
10     15
11     84
Name: count, dtype: int64


For this question, I performed a small feasibility run to verify that the same recipe set can be represented in two different ways and clustered separately. I built an ingredient-based representation and a step-text-based representation for a sample of recipes, then removed rows that became empty in either view after feature construction. This left 1,974 usable recipes. The ingredient representation produced a matrix of shape (1974, 315), while the step-text representation produced a matrix of shape (1974, 4635). I then ran hierarchical agglomerative clustering on both views and compared the resulting recipe partitions using ARI and NMI.

The initial run produced ARI = 0.118 and NMI = 0.136. These are low but clearly non-zero, which is meaningful at this stage. It suggests that ingredient composition and step text are not producing identical recipe groupings, but they are also not completely unrelated. I also checked cluster size distributions as a sanity check. On the ingredient side, the largest two clusters each contained 606 recipes, while the remaining clusters were much smaller. On the step-text side, one cluster was quite large with 870 recipes, followed by clusters of 390 and 222 recipes, with the rest smaller. This shows that clustering is feasible and does not collapse completely, but it also suggests that the feature representations may be producing uneven cluster structures. Overall, this initial check supports the feasibility of the clustering-based comparison and shows that the two views can be built and compared on the same set of recipes.

---

### RQ2 plan



**Method**

Each recipe will again be represented in two ways, but now the two views are used differently. First, each recipe is treated as an unordered ingredient basket. Second, its ordered cooking steps are converted into a normalized action sequence. On the ingredient side, we will use Apriori to mine frequent ingredient itemsets. We may also inspect association rules among ingredients if that helps interpretation, but the main structure begins with frequent itemsets.

On the step side, the raw instructions will be normalized into a smaller action vocabulary such as `chop`, `mix`, `saute`, `boil`, `bake`, `chill`, and `serve`, while preserving the order of actions. Once a frequent ingredient itemset is found, that itemset defines a subset of recipes containing that ingredient combination. Within that subset, we will run a simple sequential pattern mining algorithm such as PrefixSpan or SPADE to discover frequent ordered action subsequences.

The important point is that the sequence side will not rely on one rigid full-sequence label for each recipe. Instead, it looks for recurring ordered subsequences. This makes the method less brittle. For example, recipes with `mix -> bake`, `mix -> bake -> cool`, and `mix -> bake -> frost` can all share the procedural core `mix -> bake`.

A small example is the following. Suppose `{flour, butter, sugar}` is a frequent ingredient itemset. We collect all recipes containing that itemset. Their normalized action sequences may include `mix -> bake`, `mix -> bake -> cool`, and `mix -> bake -> frost`. Sequence mining may then recover `mix -> bake` as a frequent subsequence inside that itemset-defined subset. That would suggest that this ingredient combination has a stable procedural core, even though some recipes add optional finishing steps.

The goal is to measure procedural recurrence conditional on a shared ingredient core. Once a frequent ingredient itemset is fixed, we want to know which step subsequences recur inside that recipe subset, and whether the subset is dominated by one or a few procedural patterns or spread across many different ones.

**Evaluation criteria**

On the ingredient side, the main criterion is support of the itemset. If association rules are also examined, then confidence and lift can be reported as secondary rule-quality measures. On the sequence side, the main measures will be subsequence support within each itemset-defined subset, support of the top recurring subsequence, cumulative support of the top few subsequences, and an entropy-style concentration or diversity score over the discovered procedural patterns. We can also compare subsequence frequency within a subset against its overall frequency in the full dataset so that globally generic actions do not appear artificially meaningful.

**Motivation**

Frequent Itemsets, Generating Rules, A-Priori, Improvements over A-Priori, and Evaluating Association Rules are explicitly part of the course, so the ingredient side of this question is directly course-aligned.  
Sequential pattern mining is treated as the external extension because recipe steps are ordered by nature, and the question specifically asks about recurring ordered procedural structure rather than only unordered co-occurrence. Subsequence-based analysis is preferred over exact full templates because it captures shared procedural cores more robustly and avoids treating closely related procedures as entirely unrelated.

**Feasability**

This question is feasible because it matches the natural structure of the data. Ingredients are naturally modeled as unordered baskets, while cooking steps are naturally modeled as ordered sequences. Frequent itemset mining can therefore be applied directly on ingredients, and sequential pattern mining can be applied on normalized step-action sequences within recipe subsets defined by frequent itemsets. This makes the method conceptually clear and implementable.

**Risk**

The main risk is computational cost if the support threshold is set too low, since the ingredient vocabulary is large and long-tailed. Another risk is that results on the sequence side may depend heavily on how well the cooking steps are normalized into actions. There is also a risk that very common actions such as mix or cook may dominate the discovered subsequences and reduce interpretability.


In [23]:
# !pip -q install mlxtend prefixspan

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from prefixspan import PrefixSpan
import math
import pandas as pd

rq2 = recipes.sample(8000, random_state=42).reset_index(drop=True)

# ingredient baskets
te = TransactionEncoder()
Xb = te.fit(rq2["ingredients_list"]).transform(rq2["ingredients_list"])
basket_df = pd.DataFrame(Xb, columns=te.columns_)

# small support sensitivity check
for ms in [0.02, 0.01]:
    fi_tmp = apriori(basket_df, min_support=ms, use_colnames=True, max_len=3)
    fi_tmp["k"] = fi_tmp["itemsets"].apply(len)
    print({
        "min_support": ms,
        "n_itemsets": len(fi_tmp),
        "n_len2plus": int((fi_tmp["k"] >= 2).sum())
    })

# main small run
fi = apriori(basket_df, min_support=0.01, use_colnames=True, max_len=3)
fi["k"] = fi["itemsets"].apply(len)
fi = fi[fi["k"] >= 2].sort_values("support", ascending=False).reset_index(drop=True)

print("\nFrequent itemsets found:", len(fi))
display(fi.head(10))

chosen = set(fi.loc[0, "itemsets"])
subset = rq2[rq2["ingredients_list"].apply(lambda xs: chosen.issubset(set(xs)))].copy()
subset = subset[subset["actions"].str.len() >= 2]

print("Chosen itemset:", chosen)
print("Subset size:", len(subset))
print("Median action-sequence length:", subset["actions"].str.len().median())

ps = PrefixSpan(subset["actions"].tolist())
min_count = max(5, math.ceil(0.1 * len(subset)))
patterns = ps.frequent(min_count)
patterns = [(cnt, seq) for cnt, seq in patterns if len(seq) >= 2]
patterns = sorted(patterns, key=lambda x: -x[0])

print("Top recurring subsequences:")
for cnt, seq in patterns[:10]:
    print(cnt, "|", " -> ".join(seq))

{'min_support': 0.02, 'n_itemsets': 173, 'n_len2plus': 103}
{'min_support': 0.01, 'n_itemsets': 505, 'n_len2plus': 352}

Frequent itemsets found: 352


,support,itemsets,k
0,0.109125,"(butter, salt)",2
1,0.092750,"(salt, sugar)",2
2,0.080000,"(salt, pepper)",2
3,0.075625,"(eggs, salt)",2
4,0.070000,"(salt, onion)",2
5,0.067750,"(flour, salt)",2
6,0.066250,"(water, salt)",2
7,0.062625,"(butter, sugar)",2
8,0.058500,"(butter, eggs)",2
9,0.055000,"(butter, flour)",2


Chosen itemset: {'butter', 'salt'}
Subset size: 799
Median action-sequence length: 3.0
Top recurring subsequences:
521 | mix -> bake
289 | mix -> chill
259 | mix -> mix
241 | mix -> serve
225 | bake -> chill
214 | mix -> bake -> chill
167 | mix -> chop
161 | boil -> mix
155 | mix -> boil
134 | bake -> mix


For this question, I performed a small feasibility run to verify that both ingredient-side frequent itemset mining and step-side sequential pattern mining are practical on the dataset. On the ingredient side, I ran Apriori at two support thresholds to see whether the number of discovered itemsets remained manageable. At min_support = 0.02, the method returned 173 itemsets, of which 103 had length at least 2. At min_support = 0.01, the method returned 505 itemsets, of which 352 had length at least 2. This shows that Apriori is producing a reasonable number of frequent patterns on a moderate sample and is not immediately exploding at these support levels.

I then inspected the most frequent ingredient itemsets and found expected staple-driven patterns such as (butter, salt), (salt, sugar), (salt, pepper), and (eggs, salt). This is consistent with the earlier EDA finding that common pantry ingredients dominate raw frequency. To check the step-sequence side, I selected the frequent itemset {butter, salt}, which defined a subset of 799 recipes. Within this subset, the median normalized action-sequence length was 3.0, which indicates that the step sequences remain short enough for simple sequential mining to be practical. Running sequential pattern mining on this subset recovered recurring subsequences such as mix -> bake with count 521, mix -> chill with count 289, mix -> serve with count 241, and mix -> bake -> chill with count 214. These are meaningful because they show that the sequence-mining step is not empty or unstable; it is able to recover repeated procedural structure inside an ingredient-defined recipe subset. Overall, this check supports the feasibility of the combined itemset-plus-sequence approach and shows that the method can recover both ingredient patterns and recurring procedural subsequences.

---

### RQ3 plan

**Method**

We will build a recipe-similarity graph in which each node is a recipe and edges connect recipes with similar ingredient sets. Similarity will be weighted rather than based on raw overlap so that very common ingredients such as salt or butter do not dominate the graph. This could be done using weighted Jaccard similarity, cosine similarity on weighted ingredient vectors, or another overlap measure that downweights staples. After constructing the graph, we will run a community detection method to partition the graph into recipe groups. A modularity-based method such as Louvain is a practical choice because it scales well and produces interpretable graph communities.

Separately, we will treat recipe descriptions and steps as text and run topic modeling so that each recipe gets a topic-based semantic representation. The main external method here will be LDA, because it provides interpretable topics that can be examined through top words and representative recipes. Each recipe will therefore have both a graph-based community assignment and a topic-based text profile.

The comparison is between graph-structural groupings and text-semantic structure. If graph communities and text topics align well, that suggests ingredient-based structural similarity and text-based semantic themes are telling a similar story. If they do not align, that suggests the graph and the text are capturing different aspects of recipe organization.

**Evaluation criteria**

If each recipe is assigned a dominant topic, then graph communities and dominant topics can be compared using NMI or ARI. If full topic mixtures are retained, then we can compare topic distributions across graph communities and examine whether some communities have sharply concentrated topic profiles while others are more mixed. We will also evaluate topic coherence and inspect top words and representative recipes for interpretability. On the graph side, we can additionally report community size distribution and a graph quality measure such as modularity.

**Motivation**

Graph Mining is part of the course, including the transition from itemsets to graph mining and community detection, so the graph side of this question is course-aligned.  
Topic modeling is used as the external method it fits recipe descriptions and step text naturally. A recipe-to-recipe graph is used instead of an ingredient co-occurrence graph because the text side is also recipe-level, so the units of comparison stay aligned.

**Feasability**

This question is feasible because both the graph side and the text side are defined at the recipe level, so the two outputs can be compared directly. A recipe-similarity graph can be built from ingredient overlap or weighted similarity, and topic modeling can be applied to recipe descriptions and steps to obtain semantic groupings. Both parts are standard and fit the available data well.

**Risk**

The main risk is sensitivity to modeling choices. On the graph side, the community structure may change depending on how similarity is defined and how edges are constructed. On the text side, topic modeling may produce broad or weakly coherent topics if preprocessing is not strong enough. There is also a risk that graph communities and topic assignments may not align cleanly even if both are individually meaningful.

In [25]:
def identity(x):
    return x

rq3 = recipes.sample(2000, random_state=42).reset_index(drop=True)

# ingredient side for graph
vec_ing = TfidfVectorizer(
    tokenizer=identity,
    preprocessor=identity,
    token_pattern=None,
    lowercase=False,
    min_df=5
)
X_ing = vec_ing.fit_transform(rq3["ingredients_list"])

# text side for topics
vec_txt = CountVectorizer(
    stop_words="english",
    min_df=5,
    max_df=0.5
)
X_txt = vec_txt.fit_transform(rq3["doc_text"])

# remove zero rows in either view
mask = (X_ing.getnnz(axis=1) > 0) & (X_txt.getnnz(axis=1) > 0)
rq3 = rq3.loc[mask].reset_index(drop=True)
X_ing = X_ing[mask]
X_txt = X_txt[mask]

# sparse recipe graph
A = kneighbors_graph(X_ing, n_neighbors=8, mode="connectivity", metric="cosine", include_self=False)
G = nx.from_scipy_sparse_array(A)

comms = list(nx.community.greedy_modularity_communities(G))
node_to_comm = {}
for cid, comm in enumerate(comms):
    for node in comm:
        node_to_comm[node] = cid
graph_labels = np.array([node_to_comm[i] for i in range(len(rq3))])

# topic modeling
lda = LatentDirichletAllocation(n_components=8, random_state=42)
topic_mix = lda.fit_transform(X_txt)
topic_labels = topic_mix.argmax(axis=1)

avg_degree = np.mean([d for _, d in G.degree()])
n_components = nx.number_connected_components(G)

print("Rows kept:", len(rq3))
print("Graph nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())
print("Average degree:", avg_degree)
print("Connected components:", n_components)
print("Communities:", len(np.unique(graph_labels)))
print("Text matrix:", X_txt.shape)
print("NMI:", normalized_mutual_info_score(graph_labels, topic_labels))
print("ARI:", adjusted_rand_score(graph_labels, topic_labels))

Rows kept: 1991
Graph nodes: 1991 edges: 11081
Average degree: 11.131089904570567
Connected components: 1
Communities: 8
Text matrix: (1991, 2245)
NMI: 0.17491716621549966
ARI: 0.15518346344881795


For this question, I performed a small feasibility run to verify that a recipe-level graph can be built from ingredient similarity and then compared against text-derived topics. After removing rows that became empty in either the ingredient or text view, 1,991 recipes remained. Using ingredient-based similarity, I constructed a sparse recipe graph with 1,991 nodes and 11,081 edges. The graph had average degree about 11.13 and only 1 connected component, which indicates that the graph is neither fragmented into many disconnected pieces nor excessively dense. Community detection on this graph produced 8 communities, which is a reasonable non-trivial partition for comparison.

On the text side, I built a document-term matrix of shape (1991, 2245) from recipe descriptions and steps, and then ran topic modeling to assign recipes to latent semantic topics. Comparing graph communities with dominant topic assignments gave NMI = 0.175 and ARI = 0.155. These are again modest but non-zero values, which is sufficient for a feasibility check. They show that both outputs can be produced on the same recipe set and that some measurable relationship exists between graph-based grouping and text-based semantic structure. Overall, this initial run supports the feasibility of the graph-versus-topic comparison and shows that both the graph construction and topic modeling pipelines are workable on this dataset.

```
On my honor, I declare the following resources:
1. Collaborators:
None

2. Web Sources:
https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions/data

3. AI Tools:
Used ChatGPT to generate code for the initial EDA. I also used chatgpt to understand few external methodologies and options for plan creation. I also used it to paraphrase my observations and written descriptions to be formatted in an ordered way.

4. Citations: None
```